# INDNet 训练 —— Kaggle Notebook

**Detection-Oriented Low-Light Image Enhancement via Illumination-Noise Decoupling**

---

### 使用前必须做的三件事

1. 右上角 **Settings → Accelerator → GPU T4 x2**（不选就是纯 CPU，白等）
2. **Settings → Internet → On**（pip 安装依赖需要联网）
3. 右侧 **Input → Add Input**，挂载：
   - LOL 数据集（搜 `soumikrakshit/lol-dataset`）
   - 你自己的代码包（把 INDNet 打包上传为私有 Dataset，建议命名 `indnet-code`）

### 运行方式

全部跑通后，点 **Save Version → Save & Run All (Commit)**，
这样会在后台完整执行，**关掉浏览器也不会中断**，最长 12 小时。

> 本项目的训练在 T4 上约 1 小时，远低于会话上限。

In [ ]:
import subprocess
import sys

import torch

print("Python :", sys.version.split()[0])
print("Torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
print("设备数 :", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name}  {p.total_memory / 1024 ** 3:.1f} GB")
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout[:700])
else:
    print()
    print("[!] 没有检测到 GPU —— 请到右上角 Settings -> Accelerator 选择 GPU T4 x2")

In [ ]:
from pathlib import Path

IN = Path("/kaggle/input")


def tree(folder: Path, prefix: str = "", depth: int = 3, limit: int = 20):
    """打印目录树，方便确认数据集实际挂在哪一层。"""
    if depth < 0:
        return
    try:
        items = sorted(folder.iterdir(), key=lambda x: (x.is_file(), x.name))
    except (PermissionError, OSError):
        return
    shown = items[:limit]
    for i, item in enumerate(shown):
        last = i == len(shown) - 1
        branch = "└── " if last else "├── "
        print(f"{prefix}{branch}{item.name}{'/' if item.is_dir() else ''}")
        if item.is_dir():
            tree(item, prefix + ("    " if last else "│   "), depth - 1)
    if len(items) > limit:
        print(f"{prefix}... 还有 {len(items) - limit} 项")


print("/kaggle/input")
tree(IN, depth=3)

In [ ]:
import os
import shutil
from pathlib import Path

PROJ = Path("/kaggle/working/INDNet")
IN = Path("/kaggle/input")


def find_code_roots(base: Path, max_depth: int = 3):
    """递归找出所有「代码包根目录」。

    判定标准：该目录下同时存在 train.py 与 models/indnet.py。
    这样无论你给 Dataset 起什么名字，都能自动找到，不依赖命名约定。
    """
    found = []

    def scan(folder: Path, depth: int):
        try:
            if (folder / "train.py").is_file() and (folder / "models" / "indnet.py").is_file():
                found.append(folder)
                return
        except (PermissionError, OSError):
            return
        if depth > 0:
            for child in folder.iterdir():
                if child.is_dir():
                    scan(child, depth - 1)

    scan(base, max_depth)
    return found


roots = sorted(set(find_code_roots(IN)), key=lambda x: len(str(x)))

if roots:
    src = roots[0]
    if PROJ.exists():
        shutil.rmtree(PROJ)
    shutil.copytree(src, PROJ)
    print("代码来源 ->", src)
    print("已复制到 ->", PROJ)
else:
    print("[!] 未找到代码包（应同时包含 train.py 和 models/indnet.py）")
    print()
    print("当前已挂载的内容：")
    for item in sorted(IN.iterdir()):
        print("   -", item.name)
    print()
    print("请检查右侧 Input -> Add Input 里是否挂载了你的代码数据集")

if PROJ.is_dir():
    os.chdir(PROJ)
    print()
    print("工作目录：", Path.cwd())
    print("内容：", sorted(x.name for x in PROJ.iterdir()))


In [ ]:
# 只装 Kaggle 上缺的包
# 关键：绝不能重装 torch / torchvision —— Kaggle 镜像是配好 CUDA 的，
# 从 PyPI 重装会把 CUDA 版换成 CPU 版，GPU 就白开了。
# 所以这里用 requirements-kaggle.txt 而不是 requirements.txt

import sys

!{sys.executable} -m pip install -q -r requirements-kaggle.txt

import skimage
print("依赖就绪 | scikit-image", skimage.__version__)

In [ ]:
import yaml
from pathlib import Path

IN = Path("/kaggle/input")
RUN_CFG = Path("/kaggle/working/lol_v1_auto.yaml")


def find_lol_root(base: Path, max_depth: int = 5):
    """按目录结构特征定位 LOL 数据集，不依赖命名、不依赖挂载层级。

    判定依据（LOL-v1 的结构指纹）：
        某目录下同时存在 our485 与 eval15 两个子目录
    兜底依据：
        某目录下同时存在 low 与 high/normal 两个子目录（通用配对结构）
    """
    strict, loose = [], []

    def scan(folder: Path, depth: int):
        try:
            subs = [c for c in folder.iterdir() if c.is_dir()]
        except (PermissionError, OSError):
            return
        names = {c.name.lower() for c in subs}

        if {"our485", "eval15"} <= names:
            strict.append(folder)
            return
        if "low" in names and ({"high", "normal"} & names):
            loose.append(folder)
            return
        if depth > 0:
            for child in subs:
                scan(child, depth - 1)

    scan(base, max_depth)

    if strict:
        return sorted(set(strict), key=lambda x: len(str(x)))
    # loose 命中的是某个 split 目录，往上取一层更安全
    return sorted({p.parent for p in loose}, key=lambda x: len(str(x)))


roots = find_lol_root(IN)

print("按结构匹配到的候选根目录：")
for r in roots:
    print("   -", r)

if not roots:
    raise SystemExit(
        "[!] 没找到 LOL 数据集。请看上一格打印的目录树，确认 Input 是否已挂载。"
    )

lol_root = roots[0]

cfg = yaml.safe_load(Path("configs/lol_v1_kaggle.yaml").read_text(encoding="utf-8"))
cfg["data"]["root"] = str(lol_root)
cfg["experiment"]["output_dir"] = "/kaggle/working/INDNet_LOLv1"
cfg["eval"]["save_dir"] = "/kaggle/working/results/LOLv1"

RUN_CFG.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")

print()
print("LOL 数据根  ->", lol_root)
print("运行时配置  ->", RUN_CFG)
print("batch_size ->", cfg["data"]["batch_size"], "| base_channels ->", cfg["model"]["base_channels"])

In [ ]:
# 校验数据集：必须看到 our485 配对 485、eval15 配对 15
import sys

!{sys.executable} scripts/check_data.py --root {lol_root}

In [ ]:
# 冒烟测试：验证数据加载 + 前向 + 反向 + 优化器
# 看到「冒烟测试完成」再进入下一步，否则先排查报错
import sys

!{sys.executable} train.py --config {RUN_CFG} --smoke

In [ ]:
# 正式训练
# 显存不够就把 batch_size 降到 8 后重跑上一格生成配置
import sys

!{sys.executable} train.py --config {RUN_CFG}

In [ ]:
# 评测并导出结果图
# --decomp 会额外导出 光照图/反射图/噪声图，这是论文 Fig.4 的素材
import sys

!{sys.executable} test.py --config {RUN_CFG} --ckpt /kaggle/working/INDNet_LOLv1/best.pth --save-images --decomp --lpips

In [ ]:
import shutil
from pathlib import Path

OUT = Path("/kaggle/working/INDNet_LOLv1")
RES = Path("/kaggle/working/results/LOLv1")

for d in (OUT, RES):
    if d.is_dir():
        n = sum(1 for f in d.rglob("*") if f.is_file())
        mb = sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) / 1024 ** 2
        print(f"{d}  {n} 个文件  {mb:.1f} MB")

zip_path = shutil.make_archive("/kaggle/working/indnet_outputs", "zip", "/kaggle/working", "INDNet_LOLv1")
print()
print("已打包 ->", zip_path)
print("训练结束后在右侧 Output 面板可直接下载")

---

## （进阶）消融实验：验证方法是否真的有效

主实验只给一个 PSNR 数字，说明不了"两个分支有没有用"。
审稿人一定会问这个问题，必须用消融实验回答。

**建议节奏**：先跑前三个（结构消融）——它决定方法成不成立；
确认有增益后，再跑后两个（损失消融）。全部 5 个约 5 小时，在 12 小时会话上限内。

| 变体 | 光照分支 | 噪声分支 | 回答什么问题 |
|---|---|---|---|
| Baseline (U-Net) | ✗ | ✗ | 起点在哪 |
| ＋光照分支 | ✓ | ✗ | 光照解耦值多少 |
| INDNet (完整) | ✓ | ✓ | **噪声分支值多少（本文核心新意）** |
| －光照平滑损失 | ✓ | ✓ | 该损失项是否必要 |
| －噪声一致性损失 | ✓ | ✓ | 该损失项是否必要 |

> **幂等设计**：已有结果的变体会自动跳过。会话中途断了，重新执行本格即可接着跑，不会从头再来。

In [ ]:
import sys

# 结构消融（推荐先跑这三个，约 3 小时）
!{sys.executable} scripts/run_ablation.py --base {RUN_CFG} --out-root /kaggle/working/ablations --variants baseline_unet plus_illum full_indnet

In [ ]:
import sys

# 全部 5 个变体（约 5 小时）。已跑完的会自动跳过，可放心重复执行。
!{sys.executable} scripts/run_ablation.py --base {RUN_CFG} --out-root /kaggle/working/ablations

In [ ]:
from pathlib import Path

summary = Path("/kaggle/working/ablations/ablation_summary.md")
if summary.is_file():
    print(summary.read_text(encoding="utf-8"))
else:
    print("还没有消融结果，先跑上面两格中的一格。")

---

## 常见问题

**Q: 报 `CUDA out of memory`**
把上一格配置里的 `data.batch_size` 从 16 改成 8，重新执行生成配置的单元格再训练。

**Q: 训练中途会话断了怎么办？**
`train.py` 每 `save_every` 轮会存一次 checkpoint。用下面的命令续训：

```python
!{sys.executable} train.py --config {RUN_CFG} --resume /kaggle/working/INDNet_LOLv1/last.pth
```

**Q: 怎么查还剩多少 GPU 额度？**
Kaggle 右上角头像 → `Your Profile` → `Settings`，或 Notebook 里看右上角 Accelerator 旁的剩余时间提示。额度每周一重置。

**Q: 输出文件在哪？**
`/kaggle/working` 下的文件会随 Notebook 版本一起保存（上限 20 GB）。
训练产物默认写在 `/kaggle/working/INDNet_LOLv1/`。

**Q: 想在下一个 Notebook 里继续用这些结果？**
在 Input → Add Input 里搜索你**自己这个 Notebook 的名字**，把它的输出加进来即可。